# Reproduce the Ardook grant screener (Team A4, CSEN 166)

This notebook reruns our **final system** and checks that you get our numbers. It works on **Google Colab**, on **SCU's WAVE HPC**, or on any Linux machine with an NVIDIA GPU. Run the cells top to bottom.

**The final system** screens a grant listing for *Ardook Data* (profile: `docs/ardook-data-profile.md`) with **Qwen3-32B** ([Qwen/Qwen3-32B](https://huggingface.co/Qwen/Qwen3-32B)) in two steps:
1. **Eligibility checklist** (`src/run_pipeline.py`): the model answers fixed questions about the listing (who can apply, where, stage limits, ...) and rules in `src/ardook_grants/rules.py` decide whether Ardook Data can apply.
2. **Retrieval-augmented Fit judge** (`src/run_judge.py --k 16`): MiniLM finds the 16 most similar grants that were already rated, and the model rates this grant **High / Moderate / Low** with those as examples.

Final Fit = **Low** if the checklist rules the grant out, otherwise the judge's Fit. *Relevant* = Fit is not Low.

### Two modes

| `MODE` | What it does | Needs | Time |
|---|---|---|---|
| `"replay"` (default) | Re-scores the answers Qwen3-32B gave in our runs (bundled in `reproduce/bundle/cache/`) through the current code. Gives **exactly** our numbers. | CPU only (free Colab is fine) | ~2 min |
| `"live"` | Starts Qwen3-32B yourself with vLLM, asks it every question again, and compares its answers with ours. | NVIDIA GPU, see below | ~15 min quick, ~1 h full |

**GPU for live mode** (picked automatically from GPU memory; override with `MODEL`):

| GPU memory | Model served | Matches our results? |
|---|---|---|
| ≥ 70 GB (A100 80GB, H100) | `Qwen/Qwen3-32B` in bf16 | Same model and precision as our runs |
| 38–70 GB (A100 40GB, L40S, A6000) | `Qwen/Qwen3-32B-AWQ` (Qwen's 4-bit build) | Same model, 4-bit: expect a few grants to differ |
| 14–38 GB (T4, L4) | `Qwen/Qwen3-4B` | Smaller model: runs end to end, but scores will be lower |

On **Colab**: *Runtime → Change runtime type → A100 or H100 GPU* for live mode (Colab Pro / pay-as-you-go); any runtime works for replay. On **WAVE**: see `reproduce/README.md` for the Slurm commands.

In [ ]:
# ---- Settings (environment variables of the same name override these, for batch jobs) ----
import os

MODE = os.environ.get("REPRO_MODE", "replay")      # "replay" or "live"
SUBSET = os.environ.get("REPRO_SUBSET", "quick")   # live only: "quick" = first 40 grants of each set, "full" = all 478
MODEL = os.environ.get("REPRO_MODEL", "auto")      # live only: "auto", "Qwen/Qwen3-32B", "Qwen/Qwen3-32B-AWQ" or "Qwen/Qwen3-4B"
BASE_URL = os.environ.get("REPRO_BASE_URL", "")    # live only: an already-running OpenAI-compatible server (skips vLLM)
API_KEY_ENV = os.environ.get("REPRO_API_KEY_ENV", "REPRO_LLM_KEY")  # env var holding that server's key
LAUNCHER = os.environ.get("REPRO_LAUNCHER", "pip") # "pip": vLLM in its own virtualenv; "apptainer": vLLM's container (HPC)
HF_HOME = os.environ.get("HF_HOME", "")            # where model weights are stored; Qwen3-32B needs ~66 GB free
OWN_GRANTS = os.environ.get("REPRO_OWN_GRANTS", "")  # optional: path to your own grants (CSV/XLSX/JSON) to screen

REPO_URL = "https://github.com/SCU-CSEN166/ArdookGrantProject.git"
BRANCH = os.environ.get("REPRO_BRANCH", "main")

assert MODE in ("replay", "live") and SUBSET in ("quick", "full")
if HF_HOME:
    os.environ["HF_HOME"] = HF_HOME
print(f"MODE={MODE}  SUBSET={SUBSET}  MODEL={MODEL}  LAUNCHER={LAUNCHER}")

## 1. Get the code

If this notebook is already inside a copy of the repository (WAVE, a laptop), it is used as is. On Colab the repository is cloned. **The repository is private**: the cell asks for a GitHub token with read access (*GitHub → Settings → Developer settings → Fine-grained tokens*), or leave it blank to upload a `.zip` of the repository instead (GitHub's *Code → Download ZIP*). The token is used once and not saved.

In [ ]:
import getpass, subprocess, sys, zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules


def find_root(start: Path):
    for p in [start, *start.parents]:
        if (p / "src" / "ardook_grants").is_dir() and (p / "reproduce" / "repro.py").exists():
            return p
    for p in sorted(start.glob("*/src/ardook_grants")):  # an unzipped copy next to us
        if (p.parents[1] / "reproduce" / "repro.py").exists():
            return p.parents[1]
    return None


ROOT = find_root(Path.cwd())
if ROOT is None:
    base = Path("/content") if IN_COLAB else Path.cwd()
    target = base / "ArdookGrantProject"
    clone = ["git", "clone", "--quiet", "--depth", "1", "--branch", BRANCH]
    if subprocess.run(clone + [REPO_URL, str(target)], capture_output=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"}).returncode:
        token = getpass.getpass("Private repo: GitHub token (blank = upload a .zip instead): ").strip()
        if token:
            authed = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
            if subprocess.run(clone + [authed, str(target)], capture_output=True).returncode:
                raise SystemExit("Clone failed: check the token's access to the repository and the branch name.")
            subprocess.run(["git", "-C", str(target), "remote", "set-url", "origin", REPO_URL])  # drop the token
        elif IN_COLAB:
            from google.colab import files
            for name in files.upload():
                zipfile.ZipFile(name).extractall(base)
        else:
            raise SystemExit(f"Put a copy of the repository in {base} (or open this notebook from inside it) and rerun.")
    ROOT = find_root(base)
    assert ROOT, "Couldn't find the repository (looked for src/ardook_grants and reproduce/repro.py)."

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "reproduce"))
print("Repository:", ROOT)
print(subprocess.run(["git", "log", "-1", "--format=commit %h  %ad  %s", "--date=short"], capture_output=True, text=True).stdout or "(not a git checkout)")

## 2. Install the pipeline's dependencies and put the data in place

The pipeline itself only needs the light packages in `requirements.txt` (Colab already has most of them). The model server (vLLM) is installed separately in step 4, only for live mode.

The data comes from `reproduce/bundle/data/` and is copied into `data/` (git-ignored):
- `real_grants.csv`: the sponsor's Grant Tracker; rows 1–37 are rated for Ardook Data (28 have a Fit label).
- `synthetic_ardook_data.csv`: synthetic listings with fictional funders, labels fixed by the facts drawn before the text was written (split `pool` = judge examples, `test` = scored).
- `synthetic_ardook_data_alt.csv`: a second synthetic set drawn from different phrasings, to check the judge on wording it has never seen in its examples.

In [ ]:
import importlib, importlib.util

modules = ("torch", "transformers", "pandas", "openpyxl", "yaml", "sklearn")
if any(importlib.util.find_spec(m) is None for m in modules):
    installer = [sys.executable, "-m", "pip", "install", "-q"] if importlib.util.find_spec("pip") else ["uv", "pip", "install", "--python", sys.executable]
    subprocess.run(installer + ["-r", "requirements.txt"], check=True)

import pandas as pd
import repro
importlib.reload(repro)

copied = repro.prepare_data()
print("Copied into data/:", copied or "nothing (already there)")
OUT = ROOT / "outputs" / "reproduce"
pd.set_option("display.width", 200, "display.max_colwidth", 80)

GPU = repro.gpu()
print("GPU:", GPU or "none")

## 3. Replay: our Qwen3-32B answers through the current code (CPU)

This always runs: it is cheap and it is the reference that live mode is compared with. It reads only the bundled answers (no model is loaded besides the small MiniLM retriever) and must match `reproduce/bundle/expected_results.csv` exactly.

In [ ]:
replay_caches = repro.replay_caches(OUT / "replay" / "cache")
replay = {}
for ds in repro.DATASETS:
    print(f"\n===== {ds['title']} =====")
    replay[ds["key"]] = repro.run_system(ds, OUT / "replay", replay_caches, replay=True)

rows = [{"dataset": key, **r} for key, dirs in replay.items() for r in repro.score(dirs)]
replay_table = pd.DataFrame(rows)
expected_table = pd.DataFrame(repro.expected())
cols = [c for c in replay_table.columns if c not in ("dataset", "system")]
match = (replay_table[cols].astype(float).round(3).values == expected_table[cols].astype(float).round(3).values).all()
print("\nReplay matches reproduce/bundle/expected_results.csv:", "YES" if match else "NO, see the tables below")
replay_table

**How to read this.** *3-tier* compares High / Moderate / Low with the label; *relevant* is the yes/no question the TA's hidden test asked (Fit is not Low). The **final system** row is what we submit. On the synthetic test split the judge's examples come from the same fact pools, so the **held-out wording set** (`synthetic_alt_test`) is the fairer estimate. The **real** set is small (28 grants), and the sponsor's labels there are mostly Moderate/High, so precision is 1.0 by construction and recall is what matters.

(The TA's hidden test isn't bundled because its files are the TA's. With the files you can run it in step 6. This system scores accuracy 1.00 on its 10 grants, but that was measured after the TA's labels came out and showed they judge Ardook Data, not the laundry enterprise; our blind Week 8 submission scored 0.40.)

## 4. Live: serve the model (GPU)

Skipped in replay mode. Installs `vllm==0.18.0` (the version our Akash server ran) into its own virtualenv, downloads the weights from Hugging Face (66 GB for bf16 Qwen3-32B, 20 GB for AWQ) and starts an OpenAI-compatible server on this machine with the same settings as our server (8192-token context, thinking mode off, temperature 0). The first start takes 10–20 minutes; the log is at `outputs/reproduce/vllm.log`.

To use a server you already have (e.g. the team's Akash server from `deploy/akash/akash_llm.py`), set `BASE_URL` and put its key in the environment variable named by `API_KEY_ENV`.

In [ ]:
server = None
if MODE == "live":
    if BASE_URL:
        LIVE_MODEL = "Qwen/Qwen3-32B" if MODEL == "auto" else MODEL
        base_url, key_env = BASE_URL, API_KEY_ENV
        assert os.environ.get(key_env), f"Set the environment variable {key_env} to the server's API key."
    else:
        choice = repro.pick_model(GPU and GPU["memory_gb"], GPU["capability"] if GPU else 0)
        LIVE_MODEL = choice["model"] if MODEL == "auto" else MODEL
        print(choice["note"])
        assert LIVE_MODEL, "No usable GPU. Use MODE = 'replay', switch to a GPU runtime, or set BASE_URL."
        need = repro.MODEL_DISK_GB.get(LIVE_MODEL, 20)
        hf_dir = Path(os.environ.get("HF_HOME", Path.home() / ".cache" / "huggingface"))
        free = repro.free_disk_gb(hf_dir)
        print(f"Model: {LIVE_MODEL}   weights ~{need} GB, free disk at {hf_dir}: {free:.0f} GB")
        if free < need + 5:
            raise SystemExit(f"Not enough disk for {LIVE_MODEL}. Set HF_HOME to a bigger disk (WAVE: your scratch space).")
        env_dir = Path("/content/vllm-env") if IN_COLAB else ROOT / ".vllm-env"
        if LAUNCHER == "pip":
            repro.install_vllm(env_dir)
        extra = ["--dtype", "half"] if GPU["capability"] < 8 else []  # pre-Ampere GPUs (T4, V100) have no bf16
        server = repro.Server(LIVE_MODEL, extra, launcher=LAUNCHER, env_dir=env_dir, log=OUT / "vllm.log").start()
        server.wait()
        base_url, key_env = server.base_url, repro.KEY_ENV
else:
    print("Replay mode: no model server needed.")

## 5. Live: rerun the system and compare with our answers

Runs both steps on each set (`SUBSET = "quick"`: the first 40 grants of each; `"full"`: all 478 grants, about 2,900 model calls). Answers are cached in `outputs/reproduce/live/<model>/`, so if the runtime disconnects, rerunning this cell continues where it stopped.

The comparison has two parts: the scores of your run next to the replay of ours **on the same grants**, and how often your final Fit agrees with ours grant by grant. Greedy decoding on a different GPU or batch size can still change a few borderline answers, so expect high agreement rather than 100%.

In [ ]:
if MODE == "live":
    LIMIT = 40 if SUBSET == "quick" else None
    slug = LIVE_MODEL.split("/")[-1].lower()
    live_dir = OUT / "live" / slug
    live_caches = {"checklist": live_dir / "cache" / "checklist.json", "judge": live_dir / "cache" / "judge.json"}
    live, ref, rows, agree = {}, {}, [], []
    for ds in repro.DATASETS:
        print(f"\n===== {ds['title']} =====")
        live[ds["key"]] = repro.run_system(ds, live_dir, live_caches, replay=False, model=LIVE_MODEL,
                                           base_url=base_url, key_env=key_env, limit=LIMIT)
        ref[ds["key"]] = repro.run_system(ds, OUT / "replay_subset", replay_caches, replay=True, limit=LIMIT)
        for who, dirs in ((f"live {slug}", live[ds["key"]]), ("ours (replay)", ref[ds["key"]])):
            rows += [{"dataset": ds["key"], "run": who, **r} for r in repro.score(dirs) if r["system"] == "final system"]
        a = repro.agreement(live[ds["key"]], ref[ds["key"]])
        agree.append({"dataset": ds["key"], **{k: v for k, v in a.items() if k != "differs"}})
        for gid, program, mine, ours in a["differs"]:
            print(f"  differs: grant {gid} {program:40s} live={mine:8s} ours={ours}")
    live_table = pd.DataFrame(rows)
    repro.save_json({"model": LIVE_MODEL, "subset": SUBSET, "scores": rows, "agreement": agree}, live_dir / "summary.json")
    display(pd.DataFrame(agree))
    display(live_table)
else:
    print("Replay mode: set MODE = 'live' (first cell) on a GPU runtime to rerun the model.")

## 6. Optional: the TA's hidden test, or your own grants

- **Hidden test:** put the TA's `Ardook_hidden_test_grants.xlsx` and `ardook_hidden_test_labels.xlsx` in `hiddenTest/` (git-ignored). Needs live mode, because the model's answers for these grants aren't bundled.
- **Your own grants:** set `OWN_GRANTS` to a file in the Grant Tracker's format (columns `Program`, `Type`, `Amount / Benefit`, `Deadline`, `Core Requirements`; CSV or XLSX), a JSON list, or any CSV (see `src/run_pipeline.py --help`). The cell prints the screened list. This is what peer testers should use.

In [ ]:
hidden = ROOT / "hiddenTest" / "Ardook_hidden_test_grants.xlsx"
labels = ROOT / "hiddenTest" / "ardook_hidden_test_labels.xlsx"
extra_sets = []
if hidden.exists():
    extra_sets.append({"key": "hidden_test", "title": "TA hidden test", "input": str(hidden.relative_to(ROOT)),
                       "split": None, "labeled_only": False})
if OWN_GRANTS:
    extra_sets.append({"key": "own_grants", "title": f"Your grants ({OWN_GRANTS})", "input": OWN_GRANTS,
                       "split": None, "labeled_only": False})

if extra_sets and MODE != "live":
    print("Found", [s["key"] for s in extra_sets], "but new grants need live mode (a model to answer).")
elif not extra_sets:
    print("Nothing to do: no hiddenTest/ files and OWN_GRANTS is empty.")
for ds in extra_sets if MODE == "live" else []:
    print(f"\n===== {ds['title']} =====")
    dirs = repro.run_system(ds, live_dir, live_caches, replay=False, model=LIVE_MODEL, base_url=base_url, key_env=key_env)
    final = repro.final_predictions(dirs)
    display(pd.DataFrame([{"id": i, **{k: v for k, v in p.items() if k != "gold"}} for i, p in final.items()]))
    if ds["key"] == "hidden_test" and labels.exists():
        repro.sh([sys.executable, "evaluation/hidden_test.py", "--eligibility-run", dirs["checklist"],
                  "--fit-run", dirs["judge"], "--labels", labels, "--out", OUT / "hidden_test_outputs.csv"])

## 7. Shut down the server

Frees the GPU. (On Colab, also *Runtime → Disconnect and delete runtime* when you're done, so you aren't billed for an idle GPU.)

In [ ]:
if server:
    server.stop()
print("Done. Outputs are in", OUT)